# Supply Chain Inventory Optimization with Stochastic Demand

This notebook is the interactive companion to the GitHub report. It explains how an `(s, S)` inventory policy turns random demand into a measurable cost-service decision. The full reproducible implementation is in `src/inventory_model.py`.

## Research question

How should a single-product retailer choose a reorder point `s` and an order-up-to level `S` when daily demand is uncertain and ordering, holding, and shortage costs matter?

The model uses periodic daily review, zero lead time, and lost sales. Under a fixed policy, tomorrow's inventory depends only on today's inventory and the new demand draw, so the inventory process is a finite Markov chain.

In [ ]:
from pathlib import Path
import sys

import pandas as pd
from IPython.display import Markdown, SVG, display

ROOT = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
sys.path.insert(0, str(ROOT / 'src'))

from inventory_model import (
    BASELINE_COSTS,
    BASELINE_DEMAND,
    Policy,
    build_transition_matrix,
    markov_policy_metrics,
    simulate_policy,
    stationary_distribution,
)

ROOT

## Baseline assumptions

Demand can take values from 0 to 6 units. The table below shows the stylised baseline distribution used in the report. The values are deliberately explicit so a future version can replace them with estimated probabilities from public retail data.

In [ ]:
demand_table = pd.DataFrame(
    {
        'Daily demand': BASELINE_DEMAND.values,
        'Probability': BASELINE_DEMAND.probabilities,
    }
)
cost_table = pd.DataFrame(
    {
        'Parameter': ['Fixed order cost K', 'Unit order cost c', 'Holding cost h', 'Shortage cost p'],
        'Value': [
            BASELINE_COSTS.order_fixed_cost,
            BASELINE_COSTS.order_unit_cost,
            BASELINE_COSTS.holding_cost,
            BASELINE_COSTS.shortage_cost,
        ],
    }
)
display(demand_table)
display(cost_table)
print(f'Expected daily demand: {sum(d * p for d, p in zip(BASELINE_DEMAND.values, BASELINE_DEMAND.probabilities)):.2f} units')

## Simulate one operational trace

The baseline illustration uses policy `(3, 8)`. It is not the final recommendation; it is chosen because its order cycles are easy to inspect in a short trace. Each row shows the opening inventory, the demand draw, the order if any, ending inventory, and cost components.

In [ ]:
baseline_policy = Policy(reorder_point=3, order_up_to=8)
trace = simulate_policy(baseline_policy, periods=20, seed=7)
trace.head(12)

In [ ]:
display(SVG(filename=str(ROOT / 'outputs' / 'figures' / 'inventory_path.svg')))

## Markov-chain calculation

For a fixed policy, the possible opening inventory states are `0, 1, ..., S`. The transition matrix gives the probability of moving between these states after the policy decision and a demand realisation. Its stationary distribution represents the long-run proportion of days spent in each inventory state.

In [ ]:
recommended_policy = Policy(reorder_point=1, order_up_to=12)
transition = build_transition_matrix(recommended_policy)
stationary = stationary_distribution(transition)

stationary_table = pd.DataFrame(
    {'Opening inventory state': range(len(stationary)), 'Long-run probability': stationary}
)
display(stationary_table.style.format({'Long-run probability': '{:.2%}'}))
print('All transition rows sum to:', transition.sum(axis=1).round(10).tolist())

In [ ]:
exact_metrics = markov_policy_metrics(recommended_policy)
pd.DataFrame([exact_metrics]).T.rename(columns={0: 'Exact long-run result'}).style.format('{:.4f}')

## Compare all policies

Run `python3 src/inventory_model.py` before this section if the generated CSV files are not present. The table combines the exact Markov result with 200 simulated replications for every tested policy. The simulation is a validation method, while the stationary distribution provides the exact long-run baseline value.

In [ ]:
evaluation = pd.read_csv(ROOT / 'outputs' / 'policy_evaluation_summary.csv')
columns = [
    's', 'S', 'average_daily_cost', 'stockout_rate', 'fill_rate',
    'average_ending_inventory', 'simulation_average_daily_cost',
]
evaluation.loc[:4, columns].style.format(
    {
        'average_daily_cost': '{:.2f}',
        'stockout_rate': '{:.2%}',
        'fill_rate': '{:.2%}',
        'average_ending_inventory': '{:.2f}',
        'simulation_average_daily_cost': '{:.2f}',
    }
)

In [ ]:
display(SVG(filename=str(ROOT / 'outputs' / 'figures' / 'policy_cost_heatmap.svg')))
display(SVG(filename=str(ROOT / 'outputs' / 'figures' / 'inventory_stockout_tradeoff.svg')))

## Sensitivity analysis

The lowest-cost policy changes when the business changes how it values inventory and missed demand. A higher shortage penalty makes earlier replenishment more attractive; a higher holding cost rewards leaner inventory targets. This is why the recommended policy should always be reported with its assumptions.

In [ ]:
sensitivity = pd.read_csv(ROOT / 'outputs' / 'cost_sensitivity_summary.csv')
sensitivity.style.format(
    {
        'best_average_daily_cost': '{:.2f}',
        'best_stockout_rate': '{:.2%}',
        'best_fill_rate': '{:.2%}',
        'best_average_ending_inventory': '{:.2f}',
    }
)

In [ ]:
display(SVG(filename=str(ROOT / 'outputs' / 'figures' / 'cost_sensitivity.svg')))

## Takeaway

The baseline lowest-cost policy is `(1, 12)`, but it accepts a 10.69% stockout rate. Policy `(2, 12)` costs only modestly more and reduces the stockout rate to 5.93%. The model therefore supports a decision conversation instead of replacing it: the business can select a policy that reflects its service promise after seeing the cost of that choice.